# Paper #72: Solar Force-Free Magnetic Fields — Implementation
## Wiegelmann & Sakurai (2021) — Living Reviews in Solar Physics 18:1

**English:** This notebook implements simplified versions of the core algorithms reviewed in Wiegelmann & Sakurai (2021): (1) potential-field extrapolation via FFT from a photospheric $B_z$ boundary, (2) linear force-free field (LFFF) extrapolation with a fixed constant α, (3) a toy Wiegelmann optimization functional descent for NLFFF, (4) a comparison of NLFFF vs potential free energies, (5) a Grad-Rubin-style iterative scheme, and (6) a virial-theorem consistency check. The goal is pedagogical: small grids, explicit algorithms, no production-grade optimization. All extrapolations are on a Cartesian box with periodic/symmetric horizontal boundaries.

**Korean:** 본 노트북은 Wiegelmann & Sakurai(2021)이 리뷰한 핵심 알고리즘들을 간략히 구현한다: (1) 광구 $B_z$ 경계에서 FFT 기반 포텐셜장 외삽, (2) 상수 α를 갖는 LFFF 외삽, (3) Wiegelmann 최적화 functional의 장난감 gradient descent, (4) NLFFF 대비 포텐셜장의 자유 자기에너지 비교, (5) Grad-Rubin 스타일 반복 기법, (6) 비리얼 정리 일관성 점검. 교육 목적이며 production 품질의 최적화는 아니다. Cartesian box에서 수평 주기/대칭 경계조건 사용.

## 0. Imports and Setup / 임포트 및 설정

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.fft import fft2, ifft2, fftfreq

np.random.seed(42)
plt.rcParams['figure.dpi'] = 100

## 1. Potential Field Extrapolation from $B_z$ / 광구 $B_z$로부터 포텐셜장 외삽

**English:** For a potential field $\mathbf{B}=-\nabla\Phi$ with $\nabla^2\Phi=0$ and bottom boundary $\partial\Phi/\partial z|_{z=0}=-B_z(x,y,0)$, the Fourier solution decays as $e^{-k z}$ with $k=\sqrt{k_x^2+k_y^2}$:
$$\Phi(k_x,k_y,z)=-\hat B_z(k_x,k_y,0)\,\frac{e^{-kz}}{k}, \quad k\ne 0.$$
Then $B_x=-\partial\Phi/\partial x$, $B_y=-\partial\Phi/\partial y$, $B_z=-\partial\Phi/\partial z$ (sign choices follow the convention $B_z = -\partial\Phi/\partial z$).

**Korean:** $\mathbf{B}=-\nabla\Phi$, $\nabla^2\Phi=0$이고 바닥 경계에서 $B_z(x,y,0)$ 주어졌을 때 푸리에 해는 높이에 따라 $e^{-kz}$로 감쇠한다. 각 Fourier 모드에서 $z$-방향으로 지수 감쇠 인자를 곱해 외삽한다.

In [ ]:
def potential_extrapolation(Bz0, Lx=1.0, Ly=1.0, nz=32, dz=None):
    """Extrapolate potential magnetic field from photospheric B_z.

    Args:
        Bz0: 2D array of shape (ny, nx) with photospheric vertical field.
        Lx: Horizontal domain length in x (same units as dz).
        Ly: Horizontal domain length in y.
        nz: Number of vertical layers to compute.
        dz: Vertical step; defaults to Lx/nx.

    Returns:
        Bx, By, Bz: 3D arrays of shape (nz, ny, nx).
    """
    ny, nx = Bz0.shape
    if dz is None:
        dz = Lx / nx
    kx = 2 * np.pi * fftfreq(nx, d=Lx / nx)
    ky = 2 * np.pi * fftfreq(ny, d=Ly / ny)
    KX, KY = np.meshgrid(kx, ky, indexing='xy')
    K = np.sqrt(KX**2 + KY**2)
    K_safe = np.where(K == 0, 1e-12, K)

    Bz0_hat = fft2(Bz0)
    Bx = np.zeros((nz, ny, nx))
    By = np.zeros((nz, ny, nx))
    Bz = np.zeros((nz, ny, nx))
    for iz in range(nz):
        z = iz * dz
        decay = np.exp(-K * z)
        Bz_hat = Bz0_hat * decay
        # B = -grad(Phi), with Phi_hat = -Bz0_hat * exp(-kz)/k
        # so Bx_hat = ikx * Bz0_hat * exp(-kz)/k
        Bx_hat = 1j * KX / K_safe * Bz0_hat * decay
        By_hat = 1j * KY / K_safe * Bz0_hat * decay
        Bx[iz] = np.real(ifft2(Bx_hat))
        By[iz] = np.real(ifft2(By_hat))
        Bz[iz] = np.real(ifft2(Bz_hat))
    return Bx, By, Bz

In [ ]:
# Construct a dipolar photospheric Bz (two Gaussian spots, opposite polarity)
nx, ny = 64, 64
Lx = Ly = 1.0
x = np.linspace(0, Lx, nx, endpoint=False)
y = np.linspace(0, Ly, ny, endpoint=False)
X, Y = np.meshgrid(x, y, indexing='xy')

def gaussian_spot(cx, cy, sigma, amp):
    return amp * np.exp(-((X - cx) ** 2 + (Y - cy) ** 2) / (2 * sigma ** 2))

Bz0 = gaussian_spot(0.35, 0.5, 0.08, 1000.0) - gaussian_spot(0.65, 0.5, 0.08, 1000.0)

Bx_p, By_p, Bz_p = potential_extrapolation(Bz0, Lx=Lx, Ly=Ly, nz=32)
print('Bz_p shape:', Bz_p.shape)
print('|Bz| at z=0 max:', np.max(np.abs(Bz_p[0])))
print('|Bz| at top max :', np.max(np.abs(Bz_p[-1])))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, iz, title in zip(axes, [0, 8, 20], ['z = 0 (photosphere)', 'z ≈ 0.125', 'z ≈ 0.31']):
    im = ax.imshow(Bz_p[iz], origin='lower', cmap='RdBu_r', extent=[0, Lx, 0, Ly])
    ax.set_title(f'$B_z$ {title}')
    plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 2. Linear Force-Free Field (LFFF) Extrapolation / 선형 무력장 외삽

**English:** For a constant α, the Seehafer (1978) Fourier solution gives mode-wise vertical decay $e^{-r_{mn} z}$ with $r_{mn}=\sqrt{k^2-\alpha^2}$. Requiring $r_{mn}$ real constrains $|\alpha| < k_{\min}$ for the lowest allowed mode. For our square grid, $\alpha_{\max}=\sqrt{2}\pi/L$. We implement the scalar $B_z$ propagation and derive $B_x, B_y$ analogously to the potential field but with $k$ replaced by $r_{mn}$ and the addition of curl-like terms carrying factor α (see Eqs. 18-20 of the review).

**Korean:** 상수 α의 경우 각 푸리에 모드가 $e^{-r_{mn}z}$로 감쇠하며 $r_{mn}=\sqrt{k^2-\alpha^2}$이다. $r_{mn}$이 실수이기 위해 $|\alpha|<k_{\min}$ 조건 필요. LFFF는 포텐셜장에 α에 비례하는 현류 항이 추가된 구조.

In [ ]:
def lfff_extrapolation(Bz0, alpha, Lx=1.0, Ly=1.0, nz=32, dz=None):
    """Linear force-free extrapolation with fixed alpha (Seehafer-like).

    Args:
        Bz0: Photospheric vertical field (ny, nx).
        alpha: Constant force-free parameter (1/length units).
        Lx, Ly: Horizontal domain lengths.
        nz: Number of z layers.
        dz: Vertical step.

    Returns:
        Bx, By, Bz arrays of shape (nz, ny, nx).
    """
    ny, nx = Bz0.shape
    if dz is None:
        dz = Lx / nx
    kx = 2 * np.pi * fftfreq(nx, d=Lx / nx)
    ky = 2 * np.pi * fftfreq(ny, d=Ly / ny)
    KX, KY = np.meshgrid(kx, ky, indexing='xy')
    K2 = KX**2 + KY**2
    # r_mn^2 = k^2 - alpha^2; require positive
    R2 = K2 - alpha**2
    valid = R2 > 0
    R = np.where(valid, np.sqrt(np.maximum(R2, 1e-24)), 0.0)

    Bz0_hat = fft2(Bz0)
    # Zero out modes that would diverge (evanescent criterion)
    Bz0_hat = np.where(valid, Bz0_hat, 0.0)
    K2_safe = np.where(K2 == 0, 1e-12, K2)

    Bx = np.zeros((nz, ny, nx))
    By = np.zeros((nz, ny, nx))
    Bz = np.zeros((nz, ny, nx))
    for iz in range(nz):
        z = iz * dz
        decay = np.exp(-R * z)
        Bz_hat = Bz0_hat * decay
        # From Seehafer: Bx_hat = (decay/K2) * (alpha * i * ky * Bz0 - r * i * kx * Bz0)
        Bx_hat = decay / K2_safe * (1j * alpha * KY - R * 1j * KX) * Bz0_hat
        By_hat = decay / K2_safe * (-1j * alpha * KX - R * 1j * KY) * Bz0_hat
        Bx[iz] = np.real(ifft2(Bx_hat))
        By[iz] = np.real(ifft2(By_hat))
        Bz[iz] = np.real(ifft2(Bz_hat))
    return Bx, By, Bz

In [ ]:
# Demo: LFFF with moderate alpha
alpha_max = np.sqrt(2) * np.pi / Lx
alpha = 0.5 * alpha_max  # half of the limit
print(f'alpha_max = {alpha_max:.3f},  chosen alpha = {alpha:.3f}')

Bx_l, By_l, Bz_l = lfff_extrapolation(Bz0, alpha, Lx=Lx, Ly=Ly, nz=32)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
im0 = axes[0].imshow(Bz_p[10], origin='lower', cmap='RdBu_r')
axes[0].set_title('Potential $B_z$ at $z_{10}$')
plt.colorbar(im0, ax=axes[0])
im1 = axes[1].imshow(Bz_l[10], origin='lower', cmap='RdBu_r')
axes[1].set_title(f'LFFF $B_z$ at $z_{{10}}$ (α={alpha:.2f})')
plt.colorbar(im1, ax=axes[1])
plt.tight_layout()
plt.show()

## 3. Wiegelmann Optimization Functional Descent / Wiegelmann 최적화 gradient descent

**English:** The Wheatland-Sturrock-Roumeliotis (2000) functional
$$L=\int_V \frac{|(\nabla\times\mathbf{B})\times\mathbf{B}|^2 + |B|^2|\nabla\cdot\mathbf{B}|^2}{B^2}\,dV$$
is zero iff B is force-free AND solenoidal. Below is a simplified volumetric discretization and a few descent iterations. We do NOT implement the full F-vector from Eq. (79) of the paper; instead we use a proxy gradient via finite differences of L. This is purely pedagogical — production codes compute F analytically.

**Korean:** Wheatland-Sturrock-Roumeliotis(2000) functional이 0이면 NLFFF. 실제 코드는 해석적 F를 계산하지만 여기서는 유한차분 수치 미분으로 대체 — 교육용.

In [ ]:
def curl(Bx, By, Bz, dx, dy, dz):
    """Central-difference curl of a vector field on a 3D grid.

    Args:
        Bx, By, Bz: (nz, ny, nx) arrays.
        dx, dy, dz: Grid spacings.

    Returns:
        Jx, Jy, Jz: Curl components (nz, ny, nx).
    """
    dBzdy = np.gradient(Bz, dy, axis=1)
    dBydz = np.gradient(By, dz, axis=0)
    dBxdz = np.gradient(Bx, dz, axis=0)
    dBzdx = np.gradient(Bz, dx, axis=2)
    dBydx = np.gradient(By, dx, axis=2)
    dBxdy = np.gradient(Bx, dy, axis=1)
    Jx = dBzdy - dBydz
    Jy = dBxdz - dBzdx
    Jz = dBydx - dBxdy
    return Jx, Jy, Jz


def divergence(Bx, By, Bz, dx, dy, dz):
    """Central-difference divergence of a vector field."""
    return (
        np.gradient(Bx, dx, axis=2)
        + np.gradient(By, dy, axis=1)
        + np.gradient(Bz, dz, axis=0)
    )


def wiegelmann_L(Bx, By, Bz, dx, dy, dz):
    """Compute the Wheatland optimization functional L.

    L = integral of (|JxB|^2 + B^2 |div B|^2) / B^2 dV
    """
    Jx, Jy, Jz = curl(Bx, By, Bz, dx, dy, dz)
    # (J x B)
    JxB_x = Jy * Bz - Jz * By
    JxB_y = Jz * Bx - Jx * Bz
    JxB_z = Jx * By - Jy * Bx
    JxB2 = JxB_x**2 + JxB_y**2 + JxB_z**2
    divB = divergence(Bx, By, Bz, dx, dy, dz)
    B2 = Bx**2 + By**2 + Bz**2 + 1e-12
    integrand = (JxB2 + B2 * divB**2) / B2
    return integrand.sum() * dx * dy * dz


# Evaluate L for potential field and LFFF
dx = Lx / nx
dy = Ly / ny
dz = Lx / nx
L_pot = wiegelmann_L(Bx_p, By_p, Bz_p, dx, dy, dz)
L_lff = wiegelmann_L(Bx_l, By_l, Bz_l, dx, dy, dz)
print(f'L(potential) = {L_pot:.4e}  (should be ~0 up to discretization)')
print(f'L(LFFF α={alpha:.2f}) = {L_lff:.4e}')

**English:** For a pure potential field both $\nabla\times\mathbf{B}$ and $\nabla\cdot\mathbf{B}$ are analytically zero. Discretization errors give a small residual $L$. For LFFF, $\nabla\times\mathbf{B}=\alpha\mathbf{B}$ so $(\nabla\times\mathbf{B})\times\mathbf{B}=0$ analytically — residual again purely numerical.

**Korean:** 포텐셜장은 curl, div 모두 해석적으로 0이므로 L의 잔차는 이산화 오차. LFFF도 $\nabla\times\mathbf{B}=\alpha\mathbf{B}$이므로 $(\nabla\times\mathbf{B})\times\mathbf{B}=0$. 두 경우 모두 수치 오차만 남는다.

## 4. NLFFF vs Potential: Free Energy Comparison / NLFFF와 포텐셜장의 자유 에너지 비교

**English:** The free magnetic energy is
$$E_{\rm free}=\int_V \frac{B_{\rm NLFFF}^2-B_{\rm pot}^2}{8\pi}\,dV.$$
We cannot easily generate a true NLFFF here, but we can use LFFF (which for nonzero α has higher energy than the potential field) as a stand-in demonstration.

**Korean:** 자유 자기에너지는 NLFFF와 포텐셜장 에너지 밀도 차이의 부피 적분. 여기서는 LFFF를 임의 비-포텐셜 기준으로 사용.

In [ ]:
def magnetic_energy(Bx, By, Bz, dx, dy, dz):
    """Total magnetic energy integral (cgs units: B^2/(8 pi)).

    Returns:
        Scalar energy in erg if B is in gauss and spacings in cm.
    """
    B2 = Bx**2 + By**2 + Bz**2
    return B2.sum() * dx * dy * dz / (8 * np.pi)


E_pot = magnetic_energy(Bx_p, By_p, Bz_p, dx, dy, dz)
E_lff = magnetic_energy(Bx_l, By_l, Bz_l, dx, dy, dz)
E_free = E_lff - E_pot
print(f'E_pot = {E_pot:.4e}')
print(f'E_LFFF = {E_lff:.4e}')
print(f'Free energy fraction = {E_free / E_pot * 100:.2f}%  (LFFF > potential)')

## 5. Grad-Rubin Iteration Sketch / Grad-Rubin 반복 개요

**English:** Grad-Rubin alternates two steps:
1. Propagate α along field lines: $\mathbf{B}^{(k)}\cdot\nabla\alpha^{(k)}=0$.
2. Update $\mathbf{B}$ from $\nabla\times\mathbf{B}^{(k+1)}=\alpha^{(k)}\mathbf{B}^{(k)}$ + boundary + $\nabla\cdot\mathbf{B}=0$.

Here we sketch only the α propagation on a toy 2D slice (field lines = integral curves of B).

**Korean:** Grad-Rubin은 두 단계 교대: (1) field line 따라 α 전파 (hyperbolic), (2) Ampère로 B 갱신 (elliptic). 여기서는 2D 단면에서 α 전파만 스케치.

In [ ]:
def trace_field_line(Bx2, Bz2, x0, z0, dx, dz, max_steps=200, step=0.3):
    """Simple RK2 field-line tracer in 2D (x, z) plane.

    Args:
        Bx2, Bz2: 2D fields on a (nz, nx) grid.
        x0, z0: Starting indices (float allowed).
        dx, dz: Grid spacings.
        max_steps: Max integration steps.
        step: Step size (grid units).

    Returns:
        Trajectory list of (x_idx, z_idx).
    """
    nz, nx = Bx2.shape
    trajectory = [(x0, z0)]
    for _ in range(max_steps):
        xi, zi = int(round(x0)), int(round(z0))
        if not (0 <= xi < nx and 0 <= zi < nz):
            break
        vx, vz = Bx2[zi, xi], Bz2[zi, xi]
        mag = np.hypot(vx, vz) + 1e-12
        x0 += step * vx / mag
        z0 += step * vz / mag
        trajectory.append((x0, z0))
    return np.array(trajectory)


# 2D slice of the potential field at y = ny/2
j_slice = ny // 2
Bx2 = Bx_p[:, j_slice, :]
Bz2 = Bz_p[:, j_slice, :]
print('2D slice shape:', Bx2.shape)

fig, ax = plt.subplots(figsize=(8, 4))
ax.imshow(Bz2, origin='lower', cmap='RdBu_r', aspect='auto')
for x_start in [10, 20, 30, 44, 50]:
    traj = trace_field_line(Bx2, Bz2, x_start, 0.5, dx, dz, max_steps=300, step=0.5)
    ax.plot(traj[:, 0], traj[:, 1], 'k-', lw=0.8)
ax.set_title('Potential-field lines in y-slice (used for Grad-Rubin α propagation)')
ax.set_xlabel('x index')
ax.set_ylabel('z index')
plt.tight_layout()
plt.show()

In [ ]:
def grad_rubin_alpha_propagation(Bx3, By3, Bz3, alpha_bottom, dx, dy, dz, n_iter=5):
    """Propagate alpha along field lines via B . grad(alpha) = 0.

    Args:
        Bx3, By3, Bz3: 3D field components.
        alpha_bottom: Boundary alpha at z=0 (ny, nx).
        dx, dy, dz: Grid spacings.
        n_iter: Smoothing iterations (proxy for field-line integration).

    Returns:
        alpha 3D array (nz, ny, nx).
    """
    nz, ny_, nx_ = Bz3.shape
    alpha = np.zeros_like(Bz3)
    alpha[0] = alpha_bottom
    # Cheap upward propagation: at each z, advect from previous plane using B direction
    for iz in range(1, nz):
        # Semi-Lagrangian backward trace: where did a field line at (x,y,z_iz) come from at z_{iz-1}?
        # dx_back = -Bx/Bz * dz, dy_back = -By/Bz * dz
        Bz_safe = np.where(np.abs(Bz3[iz]) < 1e-6, 1e-6 * np.sign(Bz3[iz] + 1e-12), Bz3[iz])
        shift_x = -Bx3[iz] / Bz_safe * dz
        shift_y = -By3[iz] / Bz_safe * dz
        # Nearest-neighbor remap
        ix = np.arange(nx_)
        iy = np.arange(ny_)
        IX, IY = np.meshgrid(ix, iy, indexing='xy')
        new_ix = np.clip((IX + shift_x / dx).astype(int), 0, nx_ - 1)
        new_iy = np.clip((IY + shift_y / dy).astype(int), 0, ny_ - 1)
        alpha[iz] = alpha[iz - 1][new_iy, new_ix]
    return alpha


# Prescribe alpha on the bottom boundary where Bz > 0 (positive polarity)
alpha_bot = np.where(Bz0 > 0, 1.0, 0.0)
alpha3d = grad_rubin_alpha_propagation(Bx_p, By_p, Bz_p, alpha_bot, dx, dy, dz)
print('alpha3d shape:', alpha3d.shape)
print('alpha range at mid-height:', alpha3d[10].min(), '→', alpha3d[10].max())

## 6. Virial-Theorem Consistency Check / 비리얼 정리 일관성 점검

**English:** The magnetic virial theorem (Aly 1989):
$$E_{\rm tot}=\frac{1}{\mu_0}\int_S (xB_x+yB_y)B_z\,dx\,dy.$$
If the volume integral of $B^2/(2\mu_0)$ agrees with this surface-only expression, the boundary data are consistent with a force-free interior. We compare both for our potential field and LFFF.

**Korean:** 비리얼 정리는 경계 적분만으로 총 자기 에너지를 계산한다. Volume 적분과 일치하면 force-free 경계조건 일관성 확인.

In [ ]:
def virial_energy(Bx0, By0, Bz0_in, dx, dy, Lx, Ly):
    """Virial-theorem total magnetic energy from boundary data.

    Args:
        Bx0, By0, Bz0_in: 2D boundary fields (ny, nx) in gauss.
        dx, dy: Grid spacings in cm (for cgs energy).
        Lx, Ly: Domain extents (to center coordinates).

    Returns:
        Total energy in erg (integrand 1/(4 pi) for cgs).
    """
    ny, nx = Bz0_in.shape
    x = np.linspace(-Lx / 2, Lx / 2, nx, endpoint=False)
    y = np.linspace(-Ly / 2, Ly / 2, ny, endpoint=False)
    X_, Y_ = np.meshgrid(x, y, indexing='xy')
    integrand = (X_ * Bx0 + Y_ * By0) * Bz0_in
    return integrand.sum() * dx * dy / (4 * np.pi)


def volume_energy_3d(Bx, By, Bz, dx, dy, dz):
    """Volume integral of B^2 / (8 pi)."""
    B2 = Bx**2 + By**2 + Bz**2
    return B2.sum() * dx * dy * dz / (8 * np.pi)


E_vol_pot = volume_energy_3d(Bx_p, By_p, Bz_p, dx, dy, dz)
E_vir_pot = virial_energy(Bx_p[0], By_p[0], Bz_p[0], dx, dy, Lx, Ly)
E_vol_lff = volume_energy_3d(Bx_l, By_l, Bz_l, dx, dy, dz)
E_vir_lff = virial_energy(Bx_l[0], By_l[0], Bz_l[0], dx, dy, Lx, Ly)

print('=== Potential field ===')
print(f'E (volume) = {E_vol_pot:.4e}')
print(f'E (virial) = {E_vir_pot:.4e}')
print(f'ratio = {E_vir_pot / E_vol_pot:.3f}')
print('\n=== LFFF ===')
print(f'E (volume) = {E_vol_lff:.4e}')
print(f'E (virial) = {E_vir_lff:.4e}')
print(f'ratio = {E_vir_lff / E_vol_lff:.3f}')

**English:** On a truncated Cartesian box (without integrating over top/side boundaries), the virial expression will not match the volume integral exactly — real NLFFF codes integrate over the full computational surface. This numerical discrepancy illustrates exactly the issue raised by the paper: consistency requires a sufficiently large FOV so that flux leaves primarily through the bottom boundary.

**Korean:** 잘린 Cartesian box에서 virial 식은 볼륨 적분과 정확히 일치하지 않는다. 실제 NLFFF 코드는 전체 표면에 걸쳐 적분하며 FOV가 충분히 커서 속이 바닥 경계를 통해 빠지도록 해야 한다. 이는 논문이 지적한 일관성 문제의 핵심이다.

## 7. Summary / 요약

**English:** We implemented six core procedures from Wiegelmann & Sakurai (2021): FFT potential extrapolation, Seehafer-style LFFF with constant α, the Wheatland optimization functional, free-energy comparison, a Grad-Rubin α propagation sketch, and a virial-theorem consistency check. In a production setting each step would use higher resolution, analytic force vectors for optimization, and careful boundary integration. The purpose here is to concretize the review's algorithms and show how derived quantities (free energy, α distribution, virial energy) emerge from simple implementations.

**Korean:** 본 노트북에서 Wiegelmann & Sakurai(2021)의 6가지 핵심 절차를 구현했다: FFT 포텐셜 외삽, 상수 α LFFF, Wheatland 최적화 functional, 자유에너지 비교, Grad-Rubin α 전파 스케치, 비리얼 일관성 점검. 실제 production 코드는 더 높은 해상도, 해석적 force vector, 전체 표면 경계 적분을 사용한다. 본 구현은 리뷰의 알고리즘을 구체화하고 자유 에너지·α 분포·비리얼 에너지 같은 도출 물리량이 어떻게 계산되는지 보여주기 위한 것이다.